# Working with USB Hardware in v4.0

® *Copyright Bimea 2024-2026*

This notebook shows how v4.0 simplifies USB device management while still providing low-level access when needed.

Let's explore both the high-level (recommended) and low-level approaches.

In [ ]:
import megamicros
from megamicros.log import log

# Set log level to INFO to see device detection details
log.setLevel("INFO")

print(f"Megamicros version: {megamicros.__version__}")

## Low-Level USB Access (Advanced)

For advanced users who need direct control over USB transfers, the low-level `Usb` class is available.

**⚠ Warning**: Only use this if configure custom USB behaviors. Most users should use the high-level API above.

In what follows you can check your hardware megamicros device. 

In [ ]:
from megamicros.usb import Usb

# Megamicros USB device identifiers
VENDOR_ID = 0xFE27
PRODUCT_IDS = {
    0xAC00: "Mu32-usb2",
    0xAC01: "Mu256-usb3",
    0xAC02: "Mu1024-usb3",
    0xAC03: "Mu32-usb3",
    0xAC04: "Mu64-usb3"
}

# Check for any Megamicros device
device_found = False
detected_product_ids = []

for product_id, device_name in PRODUCT_IDS.items():
    if Usb.checkDeviceByVendorProduct(vendor_id=VENDOR_ID, product_id=product_id):
        print(f"✓ Found: {device_name} (Product ID: 0x{product_id:04X})")
        device_found = True
        detected_product_ids.append(product_id)

if not device_found:
    print("⚠ No Megamicros USB device found")
    print("Available product IDs:", [f"0x{pid:04X}" for pid in PRODUCT_IDS.keys()])

### Low-Level USB Operations

If you found a device above, you can perform low-level operations:

In [ ]:
if device_found:
    detected_product_id=detected_product_ids[0]  # Use the first detected device for demonstration
    try:
        # Initialize USB connection
        usb_device = Usb()
        usb_device.open(
            vendor_id=VENDOR_ID,
            product_id=detected_product_id,
            bus_address=0x00,
            endpoint_in=0x81
        )
        
        # Claim interface for exclusive access
        usb_device.claim()
        
        print("✓ USB device opened and claimed")
        print(f"  Endpoint IN: 0x81")
        print(f"  Transfer buffer size: {usb_device.buffer_size} bytes")
        
        # Release interface when done
        usb_device.release()
        usb_device.close()
        
        print("✓ USB device released and closed")
        
    except Exception as e:
        print(f"⚠ Low-level USB error: {e}")
else:
    print("⚠ Skipping low-level operations (no device found)")

## Troubleshooting USB Connections

### Linux
In some Linux distributions, only the root user has access to the USB port, so the following message may appear:

```bash
    ...
    aborting:  LIBUSB_ERROR_ACCESS [-3]
```

```bash
sudo tee /etc/udev/rules.d/99-megamicros-devices.rules > /dev/null << 'EOF'
# Megamicros devices
SUBSYSTEM=="usb", ATTR{idVendor}=="fe27", ATTR{idProduct}=="ac00", MODE="0666"
SUBSYSTEM=="usb", ATTR{idVendor}=="fe27", ATTR{idProduct}=="ac01", MODE="0666"
SUBSYSTEM=="usb", ATTR{idVendor}=="fe27", ATTR{idProduct}=="ac02", MODE="0666"
SUBSYSTEM=="usb", ATTR{idVendor}=="fe27", ATTR{idProduct}=="ac03", MODE="0666"
SUBSYSTEM=="usb", ATTR{idVendor}=="fe27", ATTR{idProduct}=="ac04", MODE="0666"
EOF

sudo udevadm control --reload-rules
sudo udevadm trigger
```

User should be also in the ``plugdev`` group. Check the group file:

```bash
    > vi /etc/group
    ...
    plugdev:x:46:user_account_login
    ...
```

If there is no entry with your user account (``user_account_login`` above), then add your user account in the ``plugdev`` group.
Unplugg and plugg your usb device. All should be fine.

Don't forget that if you run your Python programs on a virtual machine, usb ports should be declared as accessible on your VM.

### macOS
Ensure you have the correct permissions:

```bash
# Check if device is visible
system_profiler SPUSBDataType | grep -i megamicros

# If not visible, try unplugging and replugging the device
```

### Windows
Before using the megamicros python library you must install the *Zadig* usb driver. 

### Device Already in Use
If another process is using the device:

```python
from megamicros.exception import MuException

try:
    antenna = Megamicros(usb=True)
except MuException as e:
    print(f"Error: {e}")
    print("Try closing other applications using the device")
```

## Key Takeaways

✅ **Recommended**: Use `Megamicros()` for automatic USB detection  
✅ **Explicit mode**: Use `Megamicros(usb=True)` to enforce USB  
✅ **Low-level access**: Use `megamicros.usb.Usb` for advanced control  
✅ **Hardware-free testing**: Auto-falls back to RandomDataSource  
✅ **Platform support**: Linux, macOS, Windows (with udev on Linux)  
✅ **Multi-device**: Automatically selects first available device  

**Next**: See Notebook 03 for advanced Megamicros features!